In [1]:
# Install required libraries
!pip install -q transformers accelerate nibabel opencv-python

# Clone the official SAM-2 repository
!git clone https://github.com/facebookresearch/segment-anything-2.git
%cd segment-anything-2
!pip install -e .
%cd ..

# Download the sam2_hiera_small pretrained weights (46M parameters)
!mkdir -p checkpoints
!wget -P checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt

Cloning into 'segment-anything-2'...
remote: Enumerating objects: 1096, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 1096 (delta 1), reused 0 (delta 0), pack-reused 1094 (from 2)
Receiving objects: 100% (1096/1096), 134.84 MiB | 15.87 MiB/s, done.
Resolving deltas: 100% (376/376), done.
/home/kaustav/research/project/segment-anything-2
Obtaining file:///home/kaustav/research/project/segment-anything-2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... 

In [18]:
import sys, os

# Add SAM-2 to Python path manually
sam2_path = "/kaggle/working/segment-anything-2"
if sam2_path not in sys.path:
    sys.path.insert(0, sam2_path)

# Also reinstall just in case
os.chdir(sam2_path)
os.system("pip install -e . -q")
os.chdir("/kaggle/working")

# Verify
from sam2.build_sam import build_sam2
print("SAM-2 import: OK")

SAM-2 import: OK


In [19]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Verify SAM-2 installed correctly
from sam2.build_sam import build_sam2
print("SAM-2 import: OK")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
SAM-2 import: OK


In [20]:
import os

# Search all kaggle input for your .py files
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".py"):
            print(os.path.join(root, f))

/kaggle/input/models/ssd8134/tgsam-2-train/pytorch/default/1/tgsam2_train.py
/kaggle/input/models/ssd8134/tgsam-2/pytorch/default/1/tgsam2_dataset.py
/kaggle/input/models/ssd8134/tgsam2-model/pytorch/default/1/tgsam2_model.py


In [21]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".py"):
            print(os.path.join(root, f))

/kaggle/input/models/ssd8134/tgsam-2-train/pytorch/default/1/tgsam2_train.py
/kaggle/input/models/ssd8134/tgsam-2/pytorch/default/1/tgsam2_dataset.py
/kaggle/input/models/ssd8134/tgsam2-model/pytorch/default/1/tgsam2_model.py


In [22]:
import os, shutil

# Direct file copy — exact paths
files_to_copy = {
    "/kaggle/input/models/ssd8134/tgsam-model/pytorch/default/1/tgsam2_model.py": "/kaggle/working/tgsam2_model.py",
    "/kaggle/input/models/ssd8134/tgsam-2/pytorch/default/1/tgsam2_dataset.py":     "/kaggle/working/tgsam2_dataset.py",
    "/kaggle/input/models/ssd8134/tgsam-2-train/pytorch/default/1/tgsam2_train.py": "/kaggle/working/tgsam2_train.py",
}

for src, dst in files_to_copy.items():
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"✓ Copied: {src}")
    else:
        print(f"✗ Not found: {src}")

# Verify
print("\n--- Verification ---")
for f in ["tgsam2_model.py", "tgsam2_dataset.py", "tgsam2_train.py"]:
    status = "✓" if os.path.exists(f"/kaggle/working/{f}") else "✗ MISSING"
    print(f"{status}  {f}")

✗ Not found: /kaggle/input/models/ssd8134/tgsam-model/pytorch/default/1/tgsam2_model.py
✓ Copied: /kaggle/input/models/ssd8134/tgsam-2/pytorch/default/1/tgsam2_dataset.py
✓ Copied: /kaggle/input/models/ssd8134/tgsam-2-train/pytorch/default/1/tgsam2_train.py

--- Verification ---
✓  tgsam2_model.py
✓  tgsam2_dataset.py
✓  tgsam2_train.py


In [23]:
import sys
sys.path.insert(0, '/kaggle/working')

import torch
from sam2.build_sam import build_sam2
from tgsam2_model import TextEncoder, TextEmbedding, TCVP, TTME, TGSAM2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load SAM-2 base
sam2 = build_sam2(
    config_file="configs/sam2/sam2_hiera_s.yaml",
    ckpt_path="checkpoints/sam2_hiera_small.pt",
    device=device
)

# Build TGSAM-2 adapter modules
text_encoder = TextEncoder(hidden_dim=256).to(device)
text_embed   = TextEmbedding(in_dim=256, out_dim=256).to(device)
tcvp         = TCVP(text_dim=256, feat_dims=(256, 256, 256)).to(device)
ttme         = TTME(feat_dim=256, text_dim=256).to(device)

model = TGSAM2(sam2, text_encoder, tcvp, ttme, text_embed).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/1e6:.1f}M / Total: {total/1e6:.1f}M")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable: 49.3M / Total: 158.8M


In [24]:
import sys
sys.path.insert(0, "/kaggle/working/segment-anything-2")

from sam2.build_sam import build_sam2
sam2 = build_sam2(
    "configs/sam2/sam2_hiera_s.yaml",
    "checkpoints/sam2_hiera_small.pt",
    device=device
)
print("SAM-2 loaded ✓")

SAM-2 loaded ✓


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

class TextEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
        self.tok  = AutoTokenizer.from_pretrained(name)
        self.bert = AutoModel.from_pretrained(name)
        for p in self.bert.parameters():
            p.requires_grad = False
        self.proj = nn.Linear(768, out_dim)

    def forward(self, texts, device):
        enc = self.tok(texts, padding=True, truncation=True,
                       max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            h = self.bert(**enc).last_hidden_state
        return self.proj(h)   # (B, L, 256)


class TCVP(nn.Module):
    def __init__(self, dim=256, num_heads=8):
        super().__init__()
        self.attn    = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.deconv1 = nn.ConvTranspose2d(dim, dim, 2, stride=2)
        self.deconv2 = nn.ConvTranspose2d(dim, dim, 2, stride=2)

    def forward(self, feats, T):
        f0, f1, f2 = feats[0], feats[1], feats[2]   # f2 = highest level
        B, C, H, W = f2.shape
        vis = f2.flatten(2).transpose(1, 2)          # (B, HW, C)
        out, _ = self.attn(T, vis, vis)
        delta  = out.mean(1).view(B, C, 1, 1)
        f2 = f2 + delta
        f1 = f1 + F.gelu(self.deconv1(f2))
        f0 = f0 + self.deconv2(f1)
        return [f0, f1, f2]


class TTME(nn.Module):
    def __init__(self, dim=256, text_dim=256):
        super().__init__()
        self.mask_conv = nn.Sequential(
            nn.Conv2d(1,   dim, 3, padding=1), nn.GELU(),
            nn.Conv2d(dim, dim, 3, padding=1), nn.GELU(),
        )
        self.feat_conv = nn.Conv2d(dim, dim, 1)
        self.text_proj = nn.Linear(text_dim, dim)
        self.out = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1), nn.GELU(),
            nn.Conv2d(dim, dim, 1),
        )

    def forward(self, mask, feat, T):
        m = F.interpolate(mask, size=feat.shape[2:],
                          mode="bilinear", align_corners=False)
        m = self.mask_conv(m)
        x = self.feat_conv(feat) + m
        t = self.text_proj(T.mean(1)).view(T.shape[0], -1, 1, 1)
        return self.out(x + t)


class TGSAM2(nn.Module):
    def __init__(self, sam2, text_enc, tcvp, ttme, dim=256):
        super().__init__()
        self.sam2     = sam2
        self.text_enc = text_enc
        self.tcvp     = tcvp
        self.ttme     = ttme
        self.t_prompt = nn.Linear(dim, 256)

    def forward(self, images, texts):
        device = images.device
        B      = images.shape[0]

        # 1. Text
        T = self.text_enc(texts, device)             # (B, L, 256)

        # 2. Image encoder
        with torch.no_grad():
            bb = self.sam2.image_encoder(images)

        feats = list(bb["backbone_fpn"])             # [f0, f1, f2]

        # 3. TCVP
        feats = self.tcvp(feats, T)

        # 4. Prepare decoder inputs
        img_embed      = feats[-1]                   # (B, 256, H, W)
        high_res_feats = [feats[0], feats[1]]        # required by SAM-2 decoder
        B, C, H, W     = img_embed.shape

        # 5. Prompts
        sparse = self.t_prompt(T)                    # (B, L, 256)
        dense  = torch.zeros(B, 256, H, W, device=device)

        # 6. PE
        with torch.no_grad():
            pe = self.sam2.sam_prompt_encoder.get_dense_pe()
            pe = F.interpolate(pe, size=(H, W),
                               mode="bilinear", align_corners=False)

        # 7. Mask decoder
        low_res, iou, _, _ = self.sam2.sam_mask_decoder(
            image_embeddings         = img_embed,
            image_pe                 = pe,
            sparse_prompt_embeddings = sparse,
            dense_prompt_embeddings  = dense,
            multimask_output         = False,
            repeat_image             = False,
            high_res_features        = high_res_feats,
        )

        masks = F.interpolate(low_res, size=images.shape[-2:],
                              mode="bilinear", align_corners=False)
        mem   = self.ttme(low_res, img_embed, T)
        return masks, iou, mem

print("Classes defined ✓")

Classes defined ✓


In [26]:
text_enc = TextEncoder(out_dim=256).to(device)
tcvp     = TCVP(dim=256).to(device)
ttme     = TTME(dim=256).to(device)
model    = TGSAM2(sam2, text_enc, tcvp, ttme).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable/1e6:.1f}M ✓")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable params: 48.5M ✓


In [27]:
import os, json, numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

class MedDataset(Dataset):
    def __init__(self, root, split="train", size=256, max_frames=8):
        self.root = root
        with open(f"{root}/{split}_sequences.json") as f:
            self.seqs = json.load(f)
        self.img_tf = T.Compose([
            T.Resize((size, size)), T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
        self.msk_tf = T.Compose([
            T.Resize((size,size), interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor()
        ])
        self.max_frames = max_frames

    def __len__(self): return len(self.seqs)

    def __getitem__(self, i):
        s = self.seqs[i]
        pairs = list(zip(s["frames"], s["masks"]))[:self.max_frames]
        frames, masks = [], []
        for fp, mp in pairs:
            frames.append(self.img_tf(Image.open(f"{self.root}/{fp}").convert("RGB")))
            masks.append(self.msk_tf(Image.open(f"{self.root}/{mp}").convert("L")))
        return {
            "frames": torch.stack(frames),
            "masks":  (torch.stack(masks) > 0.5).float(),
            "prompt": s["prompt"]
        }

def collate(batch):
    max_t = max(b["frames"].shape[0] for b in batch)
    fs, ms, ps = [], [], []
    for b in batch:
        t   = b["frames"].shape[0]
        pad = max_t - t
        f = torch.cat([b["frames"], b["frames"][-1:].expand(pad,-1,-1,-1)]) if pad else b["frames"]
        m = torch.cat([b["masks"],  torch.zeros(pad,*b["masks"].shape[1:])]) if pad else b["masks"]
        fs.append(f); ms.append(m); ps.append(b["prompt"])
    return {"frames": torch.stack(fs), "masks": torch.stack(ms), "prompt": ps}

# ── Create demo data (replace ROOT with your real dataset path) ──
ROOT = "/kaggle/working/demo_data"
os.makedirs(f"{ROOT}/images", exist_ok=True)
os.makedirs(f"{ROOT}/masks",  exist_ok=True)

seqs = []
for i in range(6):
    fr, ms = [], []
    for t in range(4):
        fn = f"s{i}_f{t}.png"
        Image.fromarray(np.random.randint(50,200,(256,256,3),dtype=np.uint8)).save(f"{ROOT}/images/{fn}")
        Image.fromarray((np.random.rand(256,256)>0.8).astype(np.uint8)*255).save(f"{ROOT}/masks/{fn}")
        fr.append(f"images/{fn}"); ms.append(f"masks/{fn}")
    seqs.append({"id":f"s{i}","frames":fr,"masks":ms,
                 "prompt":"The left ventricle is a round structure responsible for pumping oxygenated blood."})

with open(f"{ROOT}/train_sequences.json","w") as f: json.dump(seqs[:5], f)
with open(f"{ROOT}/test_sequences.json", "w") as f: json.dump(seqs[5:], f)

train_loader = DataLoader(MedDataset(ROOT,"train",size=256,max_frames=4),
                          batch_size=1, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(MedDataset(ROOT,"test", size=256,max_frames=4),
                          batch_size=1, shuffle=False, collate_fn=collate)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} ✓")

Train: 5 | Val: 1 ✓


In [28]:
def dice_loss(pred, tgt, smooth=1):
    p = torch.sigmoid(pred).flatten()
    t = tgt.flatten()
    return 1-(2*(p*t).sum()+smooth)/(p.sum()+t.sum()+smooth)

def seg_loss(pred, tgt):
    return 0.5*F.binary_cross_entropy_with_logits(pred,tgt) + 0.5*dice_loss(pred,tgt)

def calc_metrics(pred, tgt, thr=0.5):
    p = (torch.sigmoid(pred)>thr).float().flatten()
    t = tgt.float().flatten()
    tp = (p*t).sum().item()
    fp = (p*(1-t)).sum().item()
    fn = ((1-p)*t).sum().item()
    return {"dsc":(2*tp+1e-6)/(2*tp+fp+fn+1e-6)*100,
            "iou":(tp+1e-6)/(tp+fp+fn+1e-6)*100}

from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import time, os

def run_training(model, train_loader, val_loader,
                 epochs=30, lr=1e-4,
                 save_dir="/kaggle/working/checkpoints"):

    os.makedirs(save_dir, exist_ok=True)
    params = (list(model.tcvp.parameters()) +
              list(model.ttme.parameters()) +
              list(model.t_prompt.parameters()) +
              list(model.text_enc.proj.parameters()))

    opt    = Adam(params, lr=lr)
    sched  = StepLR(opt, step_size=10, gamma=0.5)
    scaler = torch.amp.GradScaler('cuda')
    best   = 0.0

    print(f"{'Ep':>4} {'TrLoss':>8} {'TrDSC':>8} {'VlDSC':>8} {'VlIoU':>8} {'Time':>6}")
    print("─"*50)

    for ep in range(1, epochs+1):
        t0 = time.time()

        # Train
        model.train()
        tl = td = 0.0
        for batch in train_loader:
            frames  = batch["frames"].to(device)
            gt      = batch["masks"].to(device)
            prompts = batch["prompt"]

            opt.zero_grad()
            preds = []
            for t in range(frames.shape[1]):
                with torch.amp.autocast('cuda'):
                    pred, _, _ = model(frames[:,t], prompts)
                preds.append(pred)

            preds = torch.stack(preds, dim=1)
            loss  = seg_loss(preds, gt)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()

            m   = calc_metrics(preds.detach(), gt)
            tl += loss.item()
            td += m["dsc"]

        sched.step()
        tl /= len(train_loader)
        td /= len(train_loader)

        # Validate
        model.eval()
        vd = vi = 0.0
        with torch.no_grad():
            for batch in val_loader:
                frames  = batch["frames"].to(device)
                gt      = batch["masks"].to(device)
                prompts = batch["prompt"]
                preds   = torch.stack(
                    [model(frames[:,t], prompts)[0]
                     for t in range(frames.shape[1])], dim=1)
                m   = calc_metrics(preds, gt)
                vd += m["dsc"]
                vi += m["iou"]
        vd /= len(val_loader)
        vi /= len(val_loader)

        elapsed = time.time() - t0
        print(f"{ep:>4d} {tl:>8.4f} {td:>7.2f}% {vd:>7.2f}% {vi:>7.2f}% {elapsed:>5.0f}s")

        if vd > best:
            best = vd
            torch.save({"epoch":ep,"model":model.state_dict(),"best_dsc":best},
                       f"{save_dir}/tgsam2_best.pt")
            print(f"     ✓ Checkpoint saved  DSC={best:.2f}%")

    print(f"\nTraining complete.  Best DSC = {best:.2f}%")

run_training(model, train_loader, val_loader, epochs=30, lr=1e-4)

  Ep   TrLoss    TrDSC    VlDSC    VlIoU   Time
──────────────────────────────────────────────────


RuntimeError: The size of tensor a (64) must match the size of tensor b (256) at non-singleton dimension 1